In [1]:
# Парсер ЕГРЮЛ

In [1]:
import xml.etree.ElementTree as ET
from typing import Any, Dict, List
from pprint import pprint
from datetime import datetime
import csv
import json
import re

In [2]:
def fix_invalid_xml(xml_content: str) -> str:
    """
    Исправляет типичные ошибки в XML выписках ЕГРЮЛ:
    - Неэкранированные кавычки внутри атрибутов
    - Некорректные пробелы в значениях атрибутов
    """
    # Паттерн для поиска атрибутов с неэкранированными кавычками
    # Ищем pattern типа attr="value " value" где есть пробел и кавычка
    def fix_quotes_in_attributes(text: str) -> str:
        # Заменяем двойные кавычки внутри значений атрибутов на &quot;
        # Ищем позиции между кавычками атрибута
        result = []
        i = 0
        in_attr = False
        attr_start = -1
        
        while i < len(text):
            if not in_attr:
                # Ищем начало атрибута
                match = re.search(r'([a-zA-Z_][a-zA-Z0-9_-]*)=["\']', text[i:])
                if match:
                    in_attr = True
                    attr_start = i + match.end()
                    result.append(text[i:i + match.end()])
                    i += match.end()
                    quote_char = text[i-1]  # " или '
                else:
                    result.append(text[i])
                    i += 1
            else:
                # Ищем закрывающую кавычку
                found = False
                for j in range(i, len(text)):
                    if text[j] == quote_char and (j == 0 or text[j-1] != '\\'):
                        # Нашли закрывающую кавычку
                        # Экранируем все внутренние кавычки
                        attr_value = text[i:j]
                        # Заменяем " на &quot; внутри значения
                        attr_value = attr_value.replace('"', '&quot;')
                        result.append(attr_value)
                        result.append(quote_char)
                        i = j + 1
                        in_attr = False
                        found = True
                        break
                if not found:
                    result.append(text[i])
                    i += 1
        
        return ''.join(result)
    
    # Сначала пробуем простой фикс: заменяем проблемные паттерны
    # Паттерн: кавычка + пробел + слово + кавычка
    xml_content = re.sub(
        r'("|\')([^"\']*?)\s+([^"\']+?)("|\')',
        lambda m: f'{m.group(1)}{m.group(2)} {m.group(3).replace(m.group(1), f"&quot;")}{m.group(4)}',
        xml_content
    )
    
    # Более агрессивный фикс для известных проблемных мест
    # Заменяем " ЛИДЕР" на &quot; ЛИДЕР&quot;
    xml_content = re.sub(
        r'(НаимЮЛПолн="[^"]*?)"\s+([^"]*?)"',
        r'\1&quot; \2&quot;',
        xml_content
    )
    
    xml_content = re.sub(
        r'(НаимСокр="[^"]*?)"\s+([^"]*?)"',
        r'\1&quot; \2&quot;',
        xml_content
    )
    
    return xml_content

In [3]:
def print_xml_structure(data: Dict[str, Any], indent: int = 0, max_depth: int = None):
    """
    Красивый вывод структуры XML с отступами.
    
    Args:
        data: Словарь с данными от parse_element
        indent: Текущий отступ
        max_depth: Максимальная глубина вывода (None - без ограничений)
    """
    current_depth = indent // 2
    
    if max_depth is not None and current_depth > max_depth:
        return
    
    prefix = "  " * current_depth
    
    # Выводим тег
    print(f"{prefix}📁 {data['tag']}")
    
    # Выводим атрибуты
    if data['attributes']:
        print(f"{prefix}  📋 Атрибуты:")
        for key, value in data['attributes'].items():
            print(f"{prefix}    • {key} = {value}")
    
    # Выводим текстовое содержимое
    if data['text']:
        print(f"{prefix}  💬 Текст: {data['text']}")
    
    # Рекурсивно выводим детей
    if data['children']:
        for child in data['children']:
            print_xml_structure(child, indent + 2, max_depth)

In [6]:
def parse_xml_full_safe(xml_file_path: str) -> Dict[str, Any]:
    """
    Безопасный парсинг XML с автоматическим исправлением ошибок.
    """
    
    def parse_element(element: ET.Element, depth: int = 3) -> Dict[str, Any]:
        result = {
            'tag': element.tag,
            'attributes': dict(element.attrib),
            'text': element.text.strip() if element.text and element.text.strip() else None,
            'children': []
        }
        
        for child in element:
            result['children'].append(parse_element(child, depth + 1))
        
        return result
    
    # Читаем файл
    with open(xml_file_path, 'r', encoding='utf-8') as f:
        xml_content = f.read()
    
    # Исправляем XML
    xml_content_fixed = fix_invalid_xml(xml_content)
    
    # Парсим исправленный XML
    try:
        root = ET.fromstring(xml_content_fixed)
    except ET.ParseError as e:
        # Если не помогло, пробуем альтернативный подход - читаем как текст
        print(f"⚠️ Стандартный парсинг не удался, пробуем альтернативный метод...")
        return parse_xml_as_text(xml_content)
    
    parsed_data = parse_element(root)
    return parsed_data

In [8]:
"""Основная функция для демонстрации всех методов парсинга."""
xml_file_path = "D:\Project_My\python_workshops\python_typing_pydantic\pri_1.xml"

print("=" * 80)
print("XML ПАРСЕР ВЫПИСКИ ЕГРЮЛ")
print("=" * 80)

# ========== 1. Полный рекурсивный парсинг ==========
print("\n🔍 1. ПОЛНЫЙ РЕКУРСИВНЫЙ ПАРСИНГ")
print("-" * 80)

try:
    full_data = parse_xml_full_safe(xml_file_path)
    print("✅ XML успешно распарсен")
    print(f"📊 Корневой тег: {full_data['tag']}")
    print(f"📊 Количество дочерних элементов: {len(full_data['children'])}")
    
    # Выводим структуру (первые 4 уровня глубины)
    print("\n📋 СТРУКТУРА XML (первые 4 уровня):")
    print_xml_structure(full_data, max_depth=3)
    
except FileNotFoundError:
    print(f"❌ Файл {xml_file_path} не найден!")
    # return
except ET.ParseError as e:
    print(f"❌ Ошибка парсинга XML: {e}")
    # return
    

XML ПАРСЕР ВЫПИСКИ ЕГРЮЛ

🔍 1. ПОЛНЫЙ РЕКУРСИВНЫЙ ПАРСИНГ
--------------------------------------------------------------------------------
✅ XML успешно распарсен
📊 Корневой тег: ЕГРЮЛ
📊 Количество дочерних элементов: 1

📋 СТРУКТУРА XML (первые 4 уровня):
📁 ЕГРЮЛ
  📁 СвЮЛ
    📋 Атрибуты:
      • ДатаВып = 2026-04-01
      • ОГРН = 1196820004105
      • ДатаОГРН = 2019-05-08
      • ИНН = 6829148810
      • КПП = 772701001
      • СпрОПФ = ОКОПФ
      • КодОПФ = 12300
      • ПолнНаимОПФ = Общества с ограниченной ответственностью
    📁 СвНаимЮЛ
      📋 Атрибуты:
        • НаимЮЛПолн = ОБЩЕСТВО С ОГРАНИЧЕННОЙ ОТВЕТСТВЕННОСТЬЮ " ЛИДЕР"
      📁 ГРНДата
        📋 Атрибуты:
          • ГРН = 1196820004105
          • ДатаЗаписи = 2019-05-08
      📁 СвНаимЮЛСокр
        📋 Атрибуты:
          • НаимСокр = ООО " ЛИДЕР"
    📁 СвАдресЮЛ
      📋 Атрибуты:
        • ВидАдрКлассиф = 1
      📁 СвМНЮЛ
        📋 Атрибуты:
          • ИдНом = 0c5b2444-70a0-4932-980c-b4dc0d3f02b5
      📁 СвАдрЮЛФИАС
    

<>:2: SyntaxWarning: invalid escape sequence '\P'
<>:2: SyntaxWarning: invalid escape sequence '\P'
C:\Users\alwin\AppData\Local\Temp\ipykernel_2876\1659386270.py:2: SyntaxWarning: invalid escape sequence '\P'
  xml_file_path = "D:\Project_My\python_workshops\python_typing_pydantic\pri_1.xml"
